# Shared setup
Every pipeline notebook starts with `%run ./_common`. It provides:
- the `catalog` parameter (job parameter / widget), so the same code runs against `supplychainiq_dev` or `supplychainiq`
- path constants
- `dq_check()` / `dq_finish()`: record data-quality checks in `ops.dq_results` and **fail the task** if any hard check fails

In [0]:
dbutils.widgets.text("catalog", "supplychainiq")
CAT = dbutils.widgets.get("catalog")
LANDING = f"/Volumes/{CAT}/bronze/landing"
CKPT = f"/Volumes/{CAT}/ops/checkpoints"

from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

spark.sql(f"""CREATE TABLE IF NOT EXISTS {CAT}.ops.dq_results (
  run_ts TIMESTAMP, notebook STRING, check_name STRING,
  observed DOUBLE, expected STRING, passed BOOLEAN, hard BOOLEAN)""")

_checks = []

def dq_check(name, observed, passed, expected="", hard=True):
    """Record one check. hard=True -> a failure fails the task; hard=False -> informational."""
    _checks.append((name, None if observed is None else float(observed), str(expected), bool(passed), bool(hard)))
    tag = "PASS" if passed else ("FAIL" if hard else "WARN")
    print(f"[{tag}] {name}: observed={observed} expected={expected}")

def dq_finish(notebook):
    """Write the checks to ops.dq_results, then raise if any hard check failed."""
    global _checks
    if _checks:
        (spark.createDataFrame(_checks, "check_name STRING, observed DOUBLE, expected STRING, passed BOOLEAN, hard BOOLEAN")
             .withColumn("notebook", F.lit(notebook)).withColumn("run_ts", F.current_timestamp())
             .select("run_ts", "notebook", "check_name", "observed", "expected", "passed", "hard")
             .write.mode("append").saveAsTable(f"{CAT}.ops.dq_results"))
    failed = [c[0] for c in _checks if c[4] and not c[3]]
    _checks = []
    if failed:
        raise AssertionError(f"{notebook}: {len(failed)} hard DQ check(s) failed: {failed}")
    print(f"{notebook}: all hard checks passed")

def table_count(t):
    return spark.table(t).count() if spark.catalog.tableExists(t) else 0

def with_schema_retry(fn, attempts=3):
    """Auto Loader (addNewColumns) stops once when it sees a new column; a restart picks up the evolved schema."""
    for i in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            if "UNKNOWNFIELD" in str(e).upper().replace("_", "") and i < attempts:
                print(f"attempt {i}: new column detected, schema evolved -> restarting stream")
                continue
            raise

print(f"catalog = {CAT}")